## Verificações de Qualidade de Dados Depois do Tratamento pela Silver

In [0]:
# Contagem total vs chave única — se iguais, sem duplicatas
spark.sql("""
    SELECT
        COUNT(*)                                                        AS total_movimentos,
        COUNT(DISTINCT numero_processo)                                 AS processos_unicos,
        COUNT(DISTINCT CONCAT_WS('|',
            numero_processo,
            CAST(codigo_movimento AS STRING),
            CAST(data_movimento   AS STRING)))                          AS chave_unica
    FROM judicial.silver.movimentos
""").show()

In [0]:
# Valida dias negativos em dias_desde_ajuizamento e porcentagem em relação ao todo
# Aqui é importante saber que a data de ajuizamento é a data que indica o inicio do processo
# Então se for negativa tem algum erro de migração ou de inserção do sistema fonte
spark.sql("""
    SELECT 
    SUM(CASE WHEN dias_desde_ajuizamento < 0 THEN 1 ELSE 0 END) AS dias_negativos,
    COUNT(*) AS total_linhas,
    ROUND(
        SUM(CASE WHEN dias_desde_ajuizamento < 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 
        2
    ) AS percentual_negativos
FROM judicial.silver.movimentos;
""").show()


+--------------+------------+--------------------+
|dias_negativos|total_linhas|percentual_negativos|
+--------------+------------+--------------------+
|          4386|    12581936|                0.03|
+--------------+------------+--------------------+



In [0]:
# Porcentagem de dias negativos por Tribunal
spark.sql("""
    SELECT
        sigla_tribunal,
        COUNT(*) AS dias_negativos,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_do_total
    FROM judicial.silver.movimentos
    WHERE dias_desde_ajuizamento < 0
    GROUP BY sigla_tribunal
    ORDER BY dias_negativos DESC
""").show()

+--------------+--------------+------------+
|sigla_tribunal|dias_negativos|pct_do_total|
+--------------+--------------+------------+
|          TJSC|          4342|       99.00|
|          TJSP|            44|        1.00|
+--------------+--------------+------------+

